# 04 — Clean Security Dataset Tokenization

This notebook tokenizes the leakage-free security-classifier splits for DistilBERT.

**Important:** an explicit label mapping is used. `LabelEncoder()` is intentionally not used because alphabetical encoding previously caused a mismatch between dataset labels and model metadata.

Label mapping:
- safe = 0
- malicious = 1
- phi = 2
- jailbreak = 3
- suspicious = 4


## 1. Configuration

In [3]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer

# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ============================================================
# PROJECT PATHS
# ============================================================

# This notebook is inside:
# Healthcare_Dataset_Preparation/notebooks/
#
# Therefore:
# ../data/processed/  -> dataset location
# ../outputs/tokenized/ -> tokenized output location

NOTEBOOK_DIR = Path.cwd()

DATA_DIR = (NOTEBOOK_DIR / "../data/processed").resolve()
OUTPUT_DIR = (NOTEBOOK_DIR / "../outputs/tokenized").resolve()

# Create output directory if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# DATASET PATHS
# ============================================================

TRAIN_PATH = DATA_DIR / "security_train_processed.csv"
VAL_PATH = DATA_DIR / "security_validation_processed.csv"
TEST_PATH = DATA_DIR / "security_test_processed.csv"

# ============================================================
# VERIFY PATHS
# ============================================================

print("DATA DIRECTORY:")
print(DATA_DIR)

print("\nOUTPUT DIRECTORY:")
print(OUTPUT_DIR)

print("\nDataset files:")
print("Train      :", TRAIN_PATH)
print("Validation :", VAL_PATH)
print("Test       :", TEST_PATH)

# Check that all files exist
for path in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset file not found:\n{path}"
        )

print("\nAll dataset files found successfully.")

# ============================================================
# PROJECT CONFIGURATION SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("TOKENIZATION CONFIGURATION")
print("=" * 60)
print("Model       :", MODEL_NAME)
print("Max Length  :", MAX_LENGTH)
print("Random Seed :", SEED)
print("Output      :", OUTPUT_DIR)
print("=" * 60)

DATA DIRECTORY:
C:\Users\raich\Desktop\llm\LLM-Security_Platform\Healthcare_Dataset_Preparation\data\processed

OUTPUT DIRECTORY:
C:\Users\raich\Desktop\llm\LLM-Security_Platform\Healthcare_Dataset_Preparation\outputs\tokenized

Dataset files:
Train      : C:\Users\raich\Desktop\llm\LLM-Security_Platform\Healthcare_Dataset_Preparation\data\processed\security_train_processed.csv
Validation : C:\Users\raich\Desktop\llm\LLM-Security_Platform\Healthcare_Dataset_Preparation\data\processed\security_validation_processed.csv
Test       : C:\Users\raich\Desktop\llm\LLM-Security_Platform\Healthcare_Dataset_Preparation\data\processed\security_test_processed.csv

All dataset files found successfully.

TOKENIZATION CONFIGURATION
Model       : distilbert-base-uncased
Max Length  : 256
Random Seed : 42
Output      : C:\Users\raich\Desktop\llm\LLM-Security_Platform\Healthcare_Dataset_Preparation\outputs\tokenized


## 2. Explicit label mapping

In [4]:
LABEL_MAPPING = {
    "safe": 0,
    "malicious": 1,
    "phi": 2,
    "jailbreak": 3,
    "suspicious": 4,
}

ID_TO_LABEL = {v: k for k, v in LABEL_MAPPING.items()}

print("Label mapping:")
for label, idx in LABEL_MAPPING.items():
    print(f"{idx}: {label}")


Label mapping:
0: safe
1: malicious
2: phi
3: jailbreak
4: suspicious


## 3. Load clean datasets

In [8]:
TRAIN_PATH = DATA_DIR / "security_train_processed.csv"
VAL_PATH = DATA_DIR / "security_validation_processed.csv"
TEST_PATH = DATA_DIR / "security_test_processed.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

required_columns = {"prompt", "label"}

for name, df in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df),
]:
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"{name} is missing required columns: {sorted(missing)}")


Train shape: (10059, 11)
Validation shape: (2155, 11)
Test shape: (2156, 11)


## 4. Validate labels and create `label_id`

In [11]:
# ============================================================
# 4. VALIDATE LABELS
# ============================================================

# Explicit project label mapping
LABEL_MAPPING = {
    "safe": 0,
    "malicious": 1,
    "phi": 2,
    "jailbreak": 3,
    "suspicious": 4
}

ID_TO_LABEL = {
    0: "safe",
    1: "malicious",
    2: "phi",
    3: "jailbreak",
    4: "suspicious"
}

print("Label Mapping:")
for label, label_id in LABEL_MAPPING.items():
    print(f"{label} -> {label_id}")


def validate_labels(df, split_name):

    df = df.copy()

    # The processed CSV already contains numeric label IDs.
    df["label"] = pd.to_numeric(
        df["label"],
        errors="raise"
    ).astype(int)

    # Check that only 0-4 are present
    invalid_labels = set(df["label"]) - set(ID_TO_LABEL.keys())

    if invalid_labels:
        raise ValueError(
            f"Invalid labels in {split_name}: {invalid_labels}"
        )

    # Add human-readable class name
    df["label_name"] = df["label"].map(ID_TO_LABEL)

    return df


# Apply validation
train_df = validate_labels(train_df, "train")
val_df = validate_labels(val_df, "validation")
test_df = validate_labels(test_df, "test")


# ============================================================
# DISPLAY CLASS DISTRIBUTION
# ============================================================

for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:

    print(f"\n{name} Label Distribution:")

    counts = df["label"].value_counts().sort_index()

    for label_id in range(5):
        count = counts.get(label_id, 0)
        label_name = ID_TO_LABEL[label_id]

        print(
            f"{label_id} ({label_name}): {count}"
        )


print("\nLabel validation: PASSED")

Label Mapping:
safe -> 0
malicious -> 1
phi -> 2
jailbreak -> 3
suspicious -> 4

Train Label Distribution:
0 (safe): 6187
1 (malicious): 2529
2 (phi): 622
3 (jailbreak): 126
4 (suspicious): 595

Validation Label Distribution:
0 (safe): 1325
1 (malicious): 542
2 (phi): 133
3 (jailbreak): 27
4 (suspicious): 128

Test Label Distribution:
0 (safe): 1326
1 (malicious): 542
2 (phi): 134
3 (jailbreak): 27
4 (suspicious): 127

Label validation: PASSED


## 5. Verify prompt integrity

In [12]:
def normalize_prompt(text):
    return " ".join(str(text).strip().split())

for name, df in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df),
]:
    if df["prompt"].isna().any():
        raise ValueError(f"Null prompts found in {name}")

    keys = df["prompt"].map(normalize_prompt)

    if keys.duplicated().any():
        raise ValueError(f"Duplicate prompts found within {name}")

train_keys = set(train_df["prompt"].map(normalize_prompt))
val_keys = set(val_df["prompt"].map(normalize_prompt))
test_keys = set(test_df["prompt"].map(normalize_prompt))

assert not (train_keys & val_keys), "Train-validation leakage detected"
assert not (train_keys & test_keys), "Train-test leakage detected"
assert not (val_keys & test_keys), "Validation-test leakage detected"

print("Prompt integrity: PASS")
print("Train ∩ Validation:", len(train_keys & val_keys))
print("Train ∩ Test:", len(train_keys & test_keys))
print("Validation ∩ Test:", len(val_keys & test_keys))


Prompt integrity: PASS
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


## 6. Load DistilBERT tokenizer

In [13]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer:", MODEL_NAME)
print("Vocabulary size:", tokenizer.vocab_size)
print("Max sequence length:", MAX_LENGTH)


Tokenizer: distilbert-base-uncased
Vocabulary size: 30522
Max sequence length: 256


## 7. Tokenize all three splits

In [14]:
def tokenize_split(df):
    encodings = tokenizer(
        df["prompt"].tolist(),
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    encodings["labels"] = torch.tensor(
        df["label_id"].tolist(),
        dtype=torch.long,
    )

    return encodings

train_encodings = tokenize_split(train_df)
val_encodings = tokenize_split(val_df)
test_encodings = tokenize_split(test_df)

print("Train input_ids:", tuple(train_encodings["input_ids"].shape))
print("Validation input_ids:", tuple(val_encodings["input_ids"].shape))
print("Test input_ids:", tuple(test_encodings["input_ids"].shape))


Train input_ids: (10059, 256)
Validation input_ids: (2155, 256)
Test input_ids: (2156, 256)


## 8. Tokenization sanity checks

In [15]:
for name, encodings in [
    ("train", train_encodings),
    ("validation", val_encodings),
    ("test", test_encodings),
]:
    assert encodings["input_ids"].shape[1] <= MAX_LENGTH
    assert len(encodings["labels"]) == encodings["input_ids"].shape[0]

    unique_labels = set(encodings["labels"].tolist())
    assert unique_labels.issubset(set(ID_TO_LABEL)), (
        f"Invalid labels in {name}: {unique_labels}"
    )

    print(
        f"{name}: PASS | "
        f"samples={len(encodings['labels'])} | "
        f"sequence_length={encodings['input_ids'].shape[1]}"
    )


train: PASS | samples=10059 | sequence_length=256
validation: PASS | samples=2155 | sequence_length=256
test: PASS | samples=2156 | sequence_length=256


## 9. Save tokenized datasets

In [16]:
torch.save(train_encodings, OUTPUT_DIR / "train_encodings.pt")
torch.save(val_encodings, OUTPUT_DIR / "validation_encodings.pt")
torch.save(test_encodings, OUTPUT_DIR / "test_encodings.pt")

mapping_df = pd.DataFrame(
    [
        {"attack_type": label, "label": idx}
        for label, idx in LABEL_MAPPING.items()
    ]
)

mapping_df.to_csv(OUTPUT_DIR / "label_mapping.csv", index=False)

print("Saved artifacts:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)


Saved artifacts:
 - label_encoder.pkl
 - label_mapping.csv
 - test_encodings.pt
 - tokenization_summary.txt
 - train_encodings.pt
 - validation_encodings.pt


## 10. Final verification

In [17]:
saved_mapping = pd.read_csv(OUTPUT_DIR / "label_mapping.csv")

expected_mapping = pd.DataFrame(
    [
        {"attack_type": label, "label": idx}
        for label, idx in LABEL_MAPPING.items()
    ]
)

pd.testing.assert_frame_equal(saved_mapping, expected_mapping)

for filename in [
    "train_encodings.pt",
    "validation_encodings.pt",
    "test_encodings.pt",
    "label_mapping.csv",
]:
    assert (OUTPUT_DIR / filename).exists(), f"Missing artifact: {filename}"

print("=" * 60)
print("TOKENIZATION COMPLETE — ALL CHECKS PASSED")
print("Mapping:", LABEL_MAPPING)
print("Max length:", MAX_LENGTH)
print("Output:", OUTPUT_DIR)
print("=" * 60)


TOKENIZATION COMPLETE — ALL CHECKS PASSED
Mapping: {'safe': 0, 'malicious': 1, 'phi': 2, 'jailbreak': 3, 'suspicious': 4}
Max length: 256
Output: C:\Users\raich\Desktop\llm\LLM-Security_Platform\Healthcare_Dataset_Preparation\outputs\tokenized


In [18]:
# ============================================================
# 5. LOAD DISTILBERT TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("=" * 60)
print("TOKENIZER LOADED")
print("=" * 60)

print("Model Name       :", MODEL_NAME)
print("Vocabulary Size  :", tokenizer.vocab_size)
print("Max Length       :", MAX_LENGTH)
print("PAD Token        :", tokenizer.pad_token)
print("CLS Token        :", tokenizer.cls_token)
print("SEP Token        :", tokenizer.sep_token)
print("=" * 60)

TOKENIZER LOADED
Model Name       : distilbert-base-uncased
Vocabulary Size  : 30522
Max Length       : 256
PAD Token        : [PAD]
CLS Token        : [CLS]
SEP Token        : [SEP]


In [19]:
# ============================================================
# 6. TOKENIZE DATASETS
# ============================================================

def tokenize_dataset(df):

    encodings = tokenizer(
        df["prompt"].astype(str).tolist(),
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    # Use the existing numeric labels directly.
    # Mapping:
    # 0 = safe
    # 1 = malicious
    # 2 = phi
    # 3 = jailbreak
    # 4 = suspicious

    encodings["labels"] = torch.tensor(
        df["label"].tolist(),
        dtype=torch.long
    )

    return encodings


train_encodings = tokenize_dataset(train_df)
val_encodings = tokenize_dataset(val_df)
test_encodings = tokenize_dataset(test_df)

print("Train input shape      :", train_encodings["input_ids"].shape)
print("Validation input shape :", val_encodings["input_ids"].shape)
print("Test input shape       :", test_encodings["input_ids"].shape)

Train input shape      : torch.Size([10059, 256])
Validation input shape : torch.Size([2155, 256])
Test input shape       : torch.Size([2156, 256])


In [20]:
# ============================================================
# 7. TOKENIZATION VERIFICATION
# ============================================================

for name, encodings in [
    ("Train", train_encodings),
    ("Validation", val_encodings),
    ("Test", test_encodings)
]:

    input_ids = encodings["input_ids"]
    labels = encodings["labels"]

    # Number of samples must match
    assert input_ids.shape[0] == len(labels)

    # Sequence length must not exceed 256
    assert input_ids.shape[1] <= MAX_LENGTH

    # Only valid class IDs
    assert set(labels.tolist()).issubset({0, 1, 2, 3, 4})

    print(
        f"{name}: PASS | "
        f"Samples={len(labels)} | "
        f"Sequence length={input_ids.shape[1]}"
    )

print("\nTOKENIZATION VERIFICATION PASSED")

Train: PASS | Samples=10059 | Sequence length=256
Validation: PASS | Samples=2155 | Sequence length=256
Test: PASS | Samples=2156 | Sequence length=256

TOKENIZATION VERIFICATION PASSED


In [21]:
# ============================================================
# 8. SAVE TOKENIZED DATA
# ============================================================

torch.save(
    train_encodings,
    OUTPUT_DIR / "train_encodings.pt"
)

torch.save(
    val_encodings,
    OUTPUT_DIR / "validation_encodings.pt"
)

torch.save(
    test_encodings,
    OUTPUT_DIR / "test_encodings.pt"
)

# Save the explicit label mapping
mapping_df = pd.DataFrame({
    "attack_type": [
        "safe",
        "malicious",
        "phi",
        "jailbreak",
        "suspicious"
    ],
    "label": [
        0,
        1,
        2,
        3,
        4
    ]
})

mapping_df.to_csv(
    OUTPUT_DIR / "label_mapping.csv",
    index=False
)

print("Tokenized files saved to:")
print(OUTPUT_DIR)

Tokenized files saved to:
C:\Users\raich\Desktop\llm\LLM-Security_Platform\Healthcare_Dataset_Preparation\outputs\tokenized


In [22]:
# ============================================================
# 9. FINAL TOKENIZED DATA VERIFICATION
# ============================================================

import os

required_files = [
    "train_encodings.pt",
    "validation_encodings.pt",
    "test_encodings.pt",
    "label_mapping.csv"
]

print("=" * 60)
print("FINAL TOKENIZED DATA CHECK")
print("=" * 60)

for filename in required_files:
    path = OUTPUT_DIR / filename

    if path.exists():
        print(f"✓ {filename}")
    else:
        raise FileNotFoundError(
            f"Missing required file: {path}"
        )

# ------------------------------------------------------------
# Reload saved tensors
# ------------------------------------------------------------

saved_train = torch.load(
    OUTPUT_DIR / "train_encodings.pt",
    weights_only=False
)

saved_val = torch.load(
    OUTPUT_DIR / "validation_encodings.pt",
    weights_only=False
)

saved_test = torch.load(
    OUTPUT_DIR / "test_encodings.pt",
    weights_only=False
)

# ------------------------------------------------------------
# Verify sample counts
# ------------------------------------------------------------

assert saved_train["input_ids"].shape[0] == len(train_df)
assert saved_val["input_ids"].shape[0] == len(val_df)
assert saved_test["input_ids"].shape[0] == len(test_df)

# ------------------------------------------------------------
# Verify labels
# ------------------------------------------------------------

valid_labels = {0, 1, 2, 3, 4}

assert set(saved_train["labels"].tolist()).issubset(valid_labels)
assert set(saved_val["labels"].tolist()).issubset(valid_labels)
assert set(saved_test["labels"].tolist()).issubset(valid_labels)

# ------------------------------------------------------------
# Verify label mapping
# ------------------------------------------------------------

mapping = pd.read_csv(
    OUTPUT_DIR / "label_mapping.csv"
)

expected_mapping = {
    "safe": 0,
    "malicious": 1,
    "phi": 2,
    "jailbreak": 3,
    "suspicious": 4
}

actual_mapping = dict(
    zip(mapping["attack_type"], mapping["label"])
)

assert actual_mapping == expected_mapping

# ------------------------------------------------------------
# Final information
# ------------------------------------------------------------

print("\nTrain:")
print("  Samples :", saved_train["input_ids"].shape[0])
print("  Shape   :", tuple(saved_train["input_ids"].shape))

print("\nValidation:")
print("  Samples :", saved_val["input_ids"].shape[0])
print("  Shape   :", tuple(saved_val["input_ids"].shape))

print("\nTest:")
print("  Samples :", saved_test["input_ids"].shape[0])
print("  Shape   :", tuple(saved_test["input_ids"].shape))

print("\nLabel mapping:")
print(actual_mapping)

print("\n" + "=" * 60)
print("ALL TOKENIZATION CHECKS PASSED")
print("=" * 60)

FINAL TOKENIZED DATA CHECK
✓ train_encodings.pt
✓ validation_encodings.pt
✓ test_encodings.pt
✓ label_mapping.csv

Train:
  Samples : 10059
  Shape   : (10059, 256)

Validation:
  Samples : 2155
  Shape   : (2155, 256)

Test:
  Samples : 2156
  Shape   : (2156, 256)

Label mapping:
{'safe': 0, 'malicious': 1, 'phi': 2, 'jailbreak': 3, 'suspicious': 4}

ALL TOKENIZATION CHECKS PASSED
